In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

from IPython.display import display

from src.annotation import (
    load_annotations,
    annotations_to_regions,
)

from src.images import (
    crop_normalized_bbox,
)

from src.lmstudio import (
    analyze_image,
    load_prompt,
    save_result,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
DATA = Path("../data")
PROMPTS = Path("../prompts")

# IMAGE_ID = "1280_AB010309_0005"
# IMAGE_ID = "00003-747-AB009300"
# IMAGE_ID = "00003-1345-AB009712"
# IMAGE_ID = "00007-97-AB008730"
# IMAGE_ID = "436_AB008999_0003"
IMAGE_ID = "670_AB009188_0003"
# IMAGE_ID = "Buchanan MSS1P9267fFA2"

source = (
    DATA
    / "images"
    / f"{IMAGE_ID}.tif"
)

annotation_path = (
    DATA
    / "ground-truth"
    / f"{IMAGE_ID}.json"
)

crop_dir = (
    DATA
    / "crops"
    / IMAGE_ID
)

crop_dir.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_8B = "qwen3-vl-8b-instruct-mlx"
MODEL_30B = "qwen3-vl-30b-a3b-instruct-mlx"

In [5]:
document = load_annotations(
    annotation_path
)

human_regions = annotations_to_regions(
    document
)

for i, region in enumerate(human_regions):
    print(
        i,
        region.id,
        region.region_type,
        region.bbox,
    )

FileNotFoundError: [Errno 2] No such file or directory: '../data/ground-truth/Buchanan MSS1P9267fFA2.json'

## Select text-bearing regions

In [ ]:
TEXT_REGION_TYPES = {
    "main_text",
    "marginal_text",
    "handwritten_annotation",
}

In [ ]:
text_regions = [
    region
    for region in human_regions
    if region.region_type in TEXT_REGION_TYPES
]

len(text_regions)

4

## Create high-resolution crops

In [ ]:
crop_records = []

for region in text_regions:

    crop = crop_normalized_bbox(
        source,
        region.bbox,
        padding=30,
    )

    crop_path = (
        crop_dir
        / f"{region.id}.png"
    )

    crop.save(crop_path)

    crop_records.append({
        "region_id": region.id,
        "region_type": region.region_type,
        "crop_path": crop_path,
    })

## Load HTR Prompt

In [ ]:
htr_prompt = load_prompt(
    PROMPTS / "htr-v1.txt"
)

## Test one crop

In [ ]:
test_record = crop_records[0]

display(
    test_record["crop_path"]
)

PosixPath('../data/crops/1280_AB010309_0005/human-382299a4.png')

In [ ]:
# Qwen 8B 

response_8b = analyze_image(
    test_record["crop_path"],
    htr_prompt,
    model=MODEL_8B,
    temperature=0.0,
    max_tokens=4096,
)

response_8b.parsed

{'script': 'Arabic',
 'language': 'Persian',
 'transcription': 'بی مCORD بی مرحم عباس باش هدیم الهب زماد شهیری\nمرحم الامی باش دالهی فادیه قسیلک داره اری کیل\nمصفف خلاصی بلك ایندیه دفع مطهر وفق',
 'confidence': 0.95}

In [ ]:
result_path = (
    DATA
    / "results"
    / IMAGE_ID
    / "qwen3-vl-8b"
    / "htr"
    / f"{test_record['region_id']}-htr-v1.json"
)

save_result(
    response_8b,
    result_path,
    task="htr",
    prompt_version="htr-v1",
)

PosixPath('../data/results/1280_AB010309_0005/qwen3-vl-8b/htr/human-382299a4-htr-v1.json')

In [ ]:
## 30B 

response_30b = analyze_image(
    test_record["crop_path"],
    htr_prompt,
    model=MODEL_30B,
    temperature=0.0,
    max_tokens=4096,
)

response_30b.parsed

{'script': 'Naskh',
 'language': 'Arabic',
 'transcription': 'كبير معدوبى مصمم عبادى باتا هيلد كر اليعب نما د شبه بارى مصمم الامى باتا والعى فاديه قديك دارته ارى كيل مصطفى خلودى بلغ اندية وفه طيه ونفى',
 'confidence': 0.95}

In [ ]:
result_path = (
    DATA
    / "results"
    / IMAGE_ID
    / "qwen3-vl-30b"
    / "htr"
    / f"{test_record['region_id']}-htr-v1.json"
)

save_result(
    response_30b,
    result_path,
    task="htr",
    prompt_version="htr-v1",
)

PosixPath('../data/results/1280_AB010309_0005/qwen3-vl-30b/htr/human-382299a4-htr-v1.json')

## Compare results

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Qwen 8B",
        "script": response_8b.parsed.get("script"),
        "language": response_8b.parsed.get("language"),
        "confidence": response_8b.parsed.get("confidence"),
        "transcription": response_8b.parsed.get("transcription"),
    },
    {
        "model": "Qwen 30B",
        "script": response_30b.parsed.get("script"),
        "language": response_30b.parsed.get("language"),
        "confidence": response_30b.parsed.get("confidence"),
        "transcription": response_30b.parsed.get("transcription"),
    },
])

comparison

,model,script,language,confidence,transcription
0,Qwen 8B,Arabic,Persian,0.95,بی مCORD بی مرحم عباس باش هدیم الهب زماد شهیری...
1,Qwen 30B,Naskh,Arabic,0.95,كبير معدوبى مصمم عبادى باتا هيلد كر اليعب نما ...
